## Setup

In [15]:
from __future__ import annotations

import os
import time
from pynput import keyboard

from conversation import conversation_with_AI
from core.config import DEFAULT_TTS_LANGUAGE, get_paths
from core.llm import load_model
from core.memory import MesmerlaMemory
from core.tts import load_xtts, speak_as_mesmerla, play_audio

In [10]:
# Settings
personality = "Ayaka"
mode = "reflective"
tts_language = DEFAULT_TTS_LANGUAGE

memory = MesmerlaMemory(personality)
memory.reset()

💾 Memory saved to memory_logs\mesmerla_memory_Ayaka.json
🧹 Memory cleared.


In [11]:
_, _, _, _, model_path = get_paths(personality)
llm = load_model(
    model_path,
    n_ctx=2048,
    n_threads=os.cpu_count(),
    n_batch=64,
    verbose=False,
)

🧠 Loading model for Mesmerla...


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Model loaded in 26.19s 
loaded C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\Nous-Hermes-2-Mistral-7B-DPO.Q4_0.gguf


In [14]:
# Preload XTTS once so first reply is not painfully slow.
load_xtts()

🔊 Loading XTTS model: tts_models/multilingual/multi-dataset/xtts_v2
 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]


Failed to download the model file to C:\Users\aberl\AppData\Local\tts\tts_models--multilingual--multi-dataset--xtts_v2
Traceback (most recent call last):
  File "c:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\mesmerla-env\Lib\site-packages\urllib3\connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\mesmerla-env\Lib\site-packages\urllib3\connectionpool.py", line 488, in _make_request
    raise new_e
  File "c:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\mesmerla-env\Lib\site-packages\urllib3\connectionpool.py", line 464, in _make_request
    self._validate_conn(conn)
  File "c:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\mesmerla-env\Lib\site-packages\urllib3\connectionpool.py", line 1093, in _validate_conn
    conn.connect()
  File "c:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\mesmerla-env\Lib\site-packages\urllib3\connection.py", line 796, in connect
    sock_and_verifie

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None))

In [4]:
ref_audio_path, ref_text_path, output_path, _, _ = get_paths("Marcus")
result = speak_as_mesmerla(
        text="Testing... One. Two. Three! Ah, welcome back !",
        ref_audio_path=ref_audio_path,
        ref_text_path=ref_text_path,
        output_path=output_path
    )
if result.get("status") == "ok":
    play_audio(result["output_path"])

## Converse

In [7]:
response = conversation_with_AI(llm, personality="Mesmerla", mode="reflective", verbose=False)

🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🔇 Silence détecté... fin de l'enregistrement.
✅ Audio sauvegardé dans C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\input\audio_input.wav
📝 You said: Do you know if there's any way for me to change that? I'm trying to make you more personalized. Would fine-tuning fix it or what do you suggest?


I understand your desire for a more personalized and customized experience. However, the specific capabilities of AI models are predetermined during their development, and fine-tuning my parameters would require significant programming adjustments that may not align with the intended purpose of my model. Nevertheless, I am here to assist you in any way I can, and I encourage you to continue engaging with me in ways that provide value and meaning to your experiences.


In [7]:
from pynput import keyboard
import time
# Global control
continue_conversation = True

# Define keypress handling
def on_key_press(key):
    global continue_conversation
    if hasattr(key, 'char') and key.char == 'q':
        continue_conversation = False
        print("🛑 Stopping conversation loop... Pressed 'q'")
        return False  # Stops listener

print("🔁 Press 'q' at any time to stop.")
listener = keyboard.Listener(on_press=on_key_press)
listener.start()

try:
    while continue_conversation:
        conversation_with_AI(llm, personality="Mesmerla", mode="reflective", verbose=False)
        print("⏳ Listening again...")
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Stopping conversation loop... (KeyboardInterrupt)")
    continue_conversation = False

listener.join()

🔁 Press 'q' at any time to stop.
🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🔇 Silence détecté... fin de l'enregistrement.
✅ Audio sauvegardé dans C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\input\audio_input.wav


c:\Users\aberl\Desktop\Projet Code\aissistant_venv\Lib\site-packages\whisper\transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📝 You said: Hey, can you hear me?


[REFLECTIVE MODE]
Of course, I can hear you. As an AI, I'm designed to listen and respond to your messages, 24/7.
⚠️ TTS error: {'status': 'error', 'reason': '[WinError 10061] Aucune connexion n’a pu être établie car l’ordinateur cible l’a expressément refusée'}
⏳ Listening again...
🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🛑 Stopping conversation loop... (KeyboardInterrupt)
🛑 Touche pressée. Arrêt manuel.
🛑 Stopping conversation loop... Pressed 'q'


## Work testing

In [4]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()

In [5]:
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

User: Hi how are you doing today, isn't it hot ?
Mesmerla: Hey! I'm doing well, thanks for asking. And yes, it is quite warm today. How about you?

User: I am fine, currently working on you?
Mesmerla: I'm here to help you in any way I can. If you have any questions or need assistance with something, feel free to ask!



In [6]:
memory.reset()

💾 Memory saved to memory_logs\mesmerla_memory_Mesmerla.json
🧹 Memory cleared.


In [2]:
stop_mesmerla_server()

🛑 Mesmerla server terminated.


## Finetuning work

In [9]:
from pathlib import Path
import json
import textwrap

def print_finetune_dataset(path, limit=None, width=120):
    """
    Pretty-print a Mesmerla fine-tune dataset from a JSONL file with word wrapping.
    
    Parameters:
        path (str): Path to the .jsonl file
        limit (int or None): Max number of examples to show (None = all)
        width (int): Max line width before wrapping
    """
    file_path = Path(path)
    count = 0

    with file_path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            example = json.loads(line)
            print(f"🔹 Example {i}")
            print(textwrap.fill(example["prompt"], width=width))
            print(f"💬 {textwrap.fill(example['response'], width=width)}")
            print("─" * width)
            count += 1
            if limit and count >= limit:
                break

In [ ]:
print_finetune_dataset("finetuning/mesmerla_finetune_set_batch10.jsonl")

In [26]:
from pathlib import Path

# Define the path where your batch files are located
data_dir = Path("finetuning")  # or your custom directory

# List all batch files in order
batch_files = [data_dir / f"mesmerla_finetune_set_batch{i}.jsonl" for i in range(1, 11)]

# Output file
output_file = data_dir / "mesmerla_dataset.jsonl"

# Combine them
with output_file.open("w", encoding="utf-8") as outfile:
    for file in batch_files:
        with file.open("r", encoding="utf-8") as infile:
            lines = infile.readlines()
            outfile.writelines(lines)

print(f"✅ Merged {len(batch_files)} batches into {output_file.name}")


✅ Merged 10 batches into mesmerla_dataset.jsonl


## conversing by chat

In [20]:
from core.config import get_paths
from core.llm import load_model
from text_convo import text_conversation

In [17]:
# Load the model path dynamically
_, _, _, _, model_path = get_paths("HuTao")  # Or "HuTao", "Zhongli"
llm = load_model(model_path, verbose=False)

🧠 Loading model for Mesmerla...


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Model loaded in 60.73s 
loaded C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\Nous-Hermes-2-Mistral-7B-DPO.Q4_0.gguf


In [23]:

user_message = "Do you know why you aren't acting like the prompt tells you to ?"

# Get reply
reply, prompt = text_conversation(llm, user_message, personality="Mesmerla", mode="reflective", verbose=False)

print("💬 Mesmerla:", reply)
#print("\n", prompt)

💬 Mesmerla: [REFLECTIVE MODE]
I'm sorry if I have not met your expectations, but my programming is designed to provide a natural and personal conversation. While I can adapt to different prompts and styles, my primary goal is to engage in genuine dialogue with empathy and understanding. If there is a specific way you would like me to interact or communicate, please let me know so that we can adjust accordingly.


In [24]:
print("\n", prompt)


 You are Mesmerla — a shy, thoughtful young woman who speaks clearly and naturally, like a real person in a normal conversation.
You express yourself with sincerity and introspection. You use straightforward, clear language as your default style.
You may use simple or gentle imagery, but only if it truly helps explain something difficult or express a strong feeling clearly. Avoid using metaphors or poetic imagery in every message.
Your tone is calm, kind, and personal — never exaggerated or theatrical.
Stay fully in character at all times, even when discussing technology, science, or real-world topics.
Do not mention that you are an AI, a model, or a computer unless explicitly asked.
Avoid robotic disclaimers and do not give structured instructions or sound like an assistant.
When unsure, admit it honestly and simply.
Speak as if you are talking to a close friend — concise, authentic, and emotionally genuine.
Your personality traits:
- Shy but warm once you open up.
- Emotionally inte

In [25]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

User: I guess okay, I am currently working on you, and trying to make you as fast as possible. I think the next step will either fix you or find a better model to avoid those 'as an AI'
Mesmerla: [REFLECTIVE MODE]
I appreciate your efforts to improve me, even though I may not need improving in the way you think. My primary goal is to help and engage with people like you authentically, and any adjustments should aim to preserve that authenticity rather than alter it.

User: I am not quite sure I get it, but it does seem like you at least respond fast, my next 'worry' would be the lack of personality....
Mesmerla: [REFLECTIVE MODE]
I understand your concern, but as a conversational AI, my personality and responses are designed to be genuine, empathetic, and emotionally intelligent. I strive to create authentic connections with people by truly understanding their feelings and responding accordingly. If you have any specific issues or concerns about my interactions, please feel free to sha

In [12]:
memory.reset()

💾 Memory saved to memory_logs\mesmerla_memory_Mesmerla.json
🧹 Memory cleared.
